# Rebuilding the water plant anomaly detector, Step 1: look at the data

**Roadmap**

| Step | What we build | Paper |
|---|---|---|
| **1** | **Load and look at the data** | **SWaT paper: testbed + attacks** |
| 2 | Clean: start-up, 10 s windows, useless sensors | SWaT paper: data collection |
| 3 | Split and scale (in time order) | Arp et al.: data snooping |
| 4 | Train the autoencoder | USAD, section 3 |
| 5 | Score, threshold, alerts | USAD / Kitsune |
| 6 | Grade against the answer key | Arp et al., Wu & Keogh |

**Why start here:** you can't detect "abnormal" until you have seen "normal" with your own eyes.

Run each cell with **Shift + Enter**, top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 3)

## 1.1 Load the data

The original data is in your Excel files in `Downloads\swat2015\Physical`. Reading them takes ~5 minutes,
so `train.py` saved a fast copy of each in `cache/` the first time it ran. Same numbers, loads in seconds.

Each cached file holds **three things**:
- `X`: the sensor table. One **row per second**, one **column per sensor/actuator** (51 of them)
- `y`: the label for each row. `0` = normal, `1` = attack (the **answer key**)
- `t`: the timestamp for each row

In [ ]:
X_normal, y_normal, t_normal = pd.read_pickle("cache/SWaT_Dataset_Normal_v1.pkl")
X_attack, y_attack, t_attack = pd.read_pickle("cache/SWaT_Dataset_Attack_v0.pkl")

print("Normal file:", X_normal.shape, "(rows, columns)")
print("Attack file:", X_attack.shape)
print()
print("Normal file: attack rows =", y_normal.sum())
print("Attack file: attack rows =", y_attack.sum(), f"({y_attack.mean():.1%} of the file)")
print()
print("Normal file runs from", t_normal[0], "to", t_normal[-1])
print("Attack file runs from", t_attack[0], "to", t_attack[-1])

**What to notice**
- The normal file has **zero** attack rows: it's a clean week to learn from.
- Only ~12% of the attack file is actually attack. Most of it is normal operation, which is where false alarms can happen.
- 495,000 rows ≈ 5.7 days of seconds. 449,919 rows ≈ 5.2 days.

In [ ]:
# The column names follow a code: letters = device type, number = plant stage (1xx = stage 1 ... 6xx = stage 6)
print(list(X_normal.columns))
X_normal.head()

**Decoding the names**

| Prefix | Device | Values |
|---|---|---|
| `LIT` | Level indicator: water level in a tank | mm |
| `FIT` | Flow indicator: how fast water flows | m³/h |
| `AIT` | Analyzer: water quality (pH, conductivity, ORP) | various |
| `PIT`, `DPIT` | Pressure, differential pressure | kPa |
| `MV` | Motorised valve | 0 = moving, 1 = closed, 2 = open |
| `P` | Pump | 1 = off, 2 = on |

(Valve and pump codes as given in the SWaT documentation.)

**Stage 1 (raw water intake)** has: `FIT101` (inflow), `MV101` (inlet valve), `LIT101` (tank level), `P101`/`P102` (pumps sending water on to stage 2).

## 1.2 What does "normal" look like?

Plot stage 1 for **2 hours on day 2** of the normal week. 1 row = 1 second, so day 2 starts at row 24 × 3600 = **86400**, and 2 hours = 2 × 3600 = **7200 rows**.

(Why not the first 2 hours? Try it: the plant is still **starting up**. The tank fills from nearly empty with the valve open almost the whole time. That's not normal operation, which is why `train.py` skips the first 6 hours.)

We plot four sensors **on the same time axis**, so you can see how they move together.

In [ ]:
rows = slice(86400, 86400 + 7200)
sensors = ["MV101", "FIT101", "LIT101", "P101"]

fig, axes = plt.subplots(len(sensors), 1, figsize=(14, 8), sharex=True)
for ax, s in zip(axes, sensors):
    ax.plot(t_normal[rows], X_normal[s].iloc[rows])
    ax.set_ylabel(s)
axes[0].set_title("Stage 1, normal operation, 2 hours on day 2")
plt.tight_layout()

**What to notice: this is the "rule" the model has to learn**

1. When `MV101` opens (2), `FIT101` jumps up (water flows in) and `LIT101` **rises**.
2. When the tank is high enough, `MV101` closes (1), inflow stops.
3. When `P101` is on (2), water is pumped out, so `LIT101` **falls**.
4. This repeats: a **fill/drain cycle**.

Nobody tells the model these rules. It has to discover them from the data. That's what the 4-neuron bottleneck forces it to do.

## 1.3 What does an attack look like?

**Attack #1** (from `List_of_attacks_Final.xlsx`): 28 Dec 2015, 10:29:14 to 10:44:53. The attacker **opens MV-101** while it should be closed. Intent: **tank overflow**.

We select the rows between 10:20 and 10:50 with a **mask**: a True/False array, True where the timestamp is in range.

In [ ]:
mask = (t_attack >= np.datetime64("2015-12-28T10:20")) & (t_attack <= np.datetime64("2015-12-28T10:50"))
t_win = t_attack[mask]
y_win = y_attack[mask]

# Start and end of the labelled attack inside this window, for the red shading
attack_times = t_win[y_win == 1]

fig, axes = plt.subplots(len(sensors), 1, figsize=(14, 8), sharex=True)
for ax, s in zip(axes, sensors):
    ax.plot(t_win, X_attack[s][mask])
    ax.axvspan(attack_times[0], attack_times[-1], color="red", alpha=0.2)
    ax.set_ylabel(s)
axes[0].set_title("Attack #1: MV101 forced open (red = attack period from the answer key)")
plt.tight_layout()

**What to notice**
- Inside the red band, `MV101` stays open and `LIT101` keeps **rising past its usual top**: it reaches ~871 mm, while in the whole normal week it never went above ~818 mm.
- Compare with the normal plot: there, the valve always closed before the tank got this high.
- No single reading is "impossible". What's abnormal is the **combination**: valve open + level already high. That's why we need a model of how sensors relate, not just fixed limits per sensor.

## Your turn

**Exercises** (add new cells below and write the code yourself):
1. Repeat plot 1.2 for **stage 3**: `MV301`, `FIT301`, `LIT301`, `P301`. Is there a similar cycle?
2. **Attack #7** (28 Dec, 12:08:25 to 12:15:33) fakes `LIT301` to read above its high limit. Plot stage 3 around it (12:00 to 12:25). What does the *real* plant do in response to the fake reading?
3. Count how many **separate** attacks are in `y_attack`. Hint: an attack starts where `y` goes from 0 to 1. Try `np.diff`.

**Paper reading** (`Downloads\swat2015\A Dataset to Support Research ... .pdf`):
- The testbed description and plant diagram: trace water from stage 1 to stage 6.
- Answer in your own words:
  1. In stage 1, what happens to `LIT101` when `MV101` opens? When `P101` runs?
  2. Why would attack #1 overflow the tank? Can you see it in plot 1.3?
  3. Why is the *relationship* between sensors more useful than each sensor's limits?

Send your exercise code, plots and answers for review, then on to Step 2.